In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("use catalog novacart")
silver_run_id=str(uuid.uuid4())
print("current silver run_id:",silver_run_id)
    


In [0]:
spark.sql("""
          create table if not exists novacart.silver.processing_control(
              
              layer string,
              entity_name string,
              last_processed_bronze_run_id string,
              last_processed_bronze_ingested_at timestamp,
              rows_merged bigint,
              run_status string,
              silver_run_id string,
              updated_at timestamp
          )
          using delta
          """)

In [0]:
def upsert_to_silver(df_source,target_table,join_key):
    if spark.catalog.tableExists(target_table):
        dt=DeltaTable.forName(spark,target_table)
        (dt.alias("target")
         .merge(df_source.alias("source"),f"target.{join_key}=source.{join_key}")
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())
    else:
        df_source.write.format("delta").saveAsTable(target_table)        


In [0]:
def get_last_processed_bronze_ingested_at(entity_name):
    ctrl=(spark.table("novacart.silver.processing_control")
.filter((col("layer")=="silver")&(col("entity_name")==entity_name) &(col("run_status")=="success"))
.orderBy(col("updated_at").desc())
.limit(1)
    )
    rows=ctrl.collect()
    if not rows:
        return None
    return rows[0]["last_processed_bronze_ingested_at"]
	
	
	

In [0]:
def upsert_silver_control(entity_name,last_processed_bronze_run_id,last_processed_bronze_ingested_at,rows_merged):
    ctrl_df=(spark.createDataFrame(
[("silver",
entity_name,
last_processed_bronze_run_id,
last_processed_bronze_ingested_at,
int(rows_merged),
"success",
silver_run_id,
datetime.now()
   )]
,"layer string, entity_name string, last_processed_bronze_run_id string, last_processed_bronze_ingested_at timestamp, rows_merged bigint, run_status string, silver_run_id string, updated_at timestamp"
))
    
    dt=DeltaTable.forName(spark,"novacart.silver.processing_control")
    (dt.alias("t").merge(ctrl_df.alias("s"),"t.layer=s.layer and t.entity_name=s.entity_name")
    .whenMatchedUpdate(set={
        "last_processed_bronze_run_id":"s.last_processed_bronze_run_id",
        "last_processed_bronze_ingested_at":"s.last_processed_bronze_ingested_at",
        "rows_merged":"s.rows_merged",
        "run_status":"s.run_status",
    "silver_run_id":"s.silver_run_id",
    "updated_at":"s.updated_at"
   })
    .whenNotMatchedInsertAll()
    .execute()
          )

In [0]:
def get_incremental_bronze(bronze_table,entity_name):
    last_ingested_at=get_last_processed_bronze_ingested_at(entity_name)
    bronze_df=spark.read.table(bronze_table)         
    if last_ingested_at is None:
        return bronze_df,last_ingested_at
    return bronze_df.filter(col("bronze_ingested_at")>lit(last_ingested_at)),last_ingested_at

In [0]:
orders_inc,last_orders_ingested_at=get_incremental_bronze("novacart.bronze.orders_raw","orders")

orders_inc_count=orders_inc.count()
print(f"orders rows_to_process_in_silver={orders_inc_count}")

if orders_inc_count>0:
    order_window=Window.partitionBy("order_id").orderBy(col("updated_at").cast("timestamp").desc(),
                                                        col("bronze_ingested_at").desc())

    orders_cleaned=(

        orders_inc.withColumn("order_status",upper(trim(col("order_status"))))
        .withColumn("order_status",when(col("order_status")=="",lit(None)).otherwise(col("order_status")))
        .withColumn("order_amount",regexp_replace(col("order_amount"),r"[$, ]",""))
        .withColumn("order_amount",when(trim(col("order_amount")).isin("N/A","NULL","??",""),None).otherwise(col("order_amount")))
        .withColumn("order_amount",col("order_amount").cast("double"))
        .withColumn("created_at",to_timestamp("created_at"))
        .withColumn("update_at",to_timestamp("updated_at"))
        .withColumn("row_rank",row_number().over(order_window))
        .filter(col("row_rank")==1)
        .drop("row_rank")
        .withColumn("silver_run_id",lit(silver_run_id))
    )     

    upsert_to_silver(orders_cleaned,"novacart.silver.orders_cleaned","order_id")

    orders_validated=(
        orders_cleaned.withColumn("to_be_verified_by_orders_team",
                                  when(col("customer_id").isNull(),"verify_customer_id")
                                  .when(col("product_id").isNull(),"verify_product_id")
                                  .when(col("order_status").isNull()|(trim(col("order_status"))==""),"verify_order_status")
                                  .when(col("order_amount").isNull()|(col("order_amount")<=0),"verify_order_amount")
                                  .otherwise("No issues")
                                                                  
                                  )
        .withColumn("check_order_amount",when(col("order_amount").isNull()|(col("order_amount")<=0),lit(True))
        .otherwise(lit(False))
    )
        .withColumn("order_date",to_date("created_at"))
        .withColumn("order_year",year("created_at"))
        .withColumn("order_month",month("created_at"))
        .withColumn("order_day",dayofmonth("created_at"))
        .withColumn("order_row",date_format("created_at","E"))
        

    )

    orders_good=orders_validated.filter(col("to_be_verified_by_orders_team")=="No issues")
    orders_bad=(

        orders_validated.filter(col("to_be_verified_by_orders_team")!="No issues")
        .withColumn("quarantine_ts",current_timestamp())
    )

    upsert_to_silver(
        orders_good,"novacart.silver.orders_transformed","order_id"
    ) 
    orders_bad.write.format("delta").mode("append").saveAsTable("novacart.silver.orders_quarantine")
    mx_ingested=orders_inc.agg(max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run=(
        orders_inc.filter(col("bronze_ingested_at")==lit(mx_ingested))
        .agg(max("bronze_run_id").alias("mx")).collect()[0]["mx"]
    )
    upsert_silver_control("orders",mx_run,mx_ingested,orders_good.count())                                                 
else:
    print("no new orders bronze rows for silver")
    upsert_silver_control(
        "orders",
        None,last_orders_ingested_at,orders_inc_count
    )

In [0]:
#PRODUCTS TABLE CLEANING

products_inc,last_products_ingested_at=get_incremental_bronze("novacart.bronze.products_raw","products")

products_inc_count=products_inc.count()
print(f"products rows to process in silver={products_inc_count}")
if products_inc_count>0:
    product_window=Window.partitionBy("product_id").orderBy(
        col("updated_at").cast("timestamp").desc(),
        col("bronze_ingested_at").desc()

    )

    products_cleaned=(
        products_inc
        .withColumn("product_name",upper(trim(col("product_name"))))
        .withColumn("product_name",regexp_replace(col("product_name"),r"[-_]"," "))
        .withColumn("product_name",when(col("product_name")=="",lit(None)).otherwise(col("product_name")))
        .withColumn("category",when(upper(trim(col("category"))).contains("ELECTRONICS"),"ELECTRONICS").otherwise(upper(trim(col("category")))))
        .withColumn("price",trim(col("price")))
        .withColumn("price",regexp_replace(col("price"),r"\$",""))
        .withColumn("price",regexp_replace(col("price"),r",","."))
        .withColumn("price",regexp_replace(col("price"),r"\s+",""))
        .withColumn("price",expr("try_cast(price as double)"))
        .withColumn("update_at",to_timestamp("updated_at"))
        .withColumn("row_rank",row_number().over(product_window))
        .filter(col("row_rank")==1)
        .drop("row_rank")
        .withColumn("silver_run_id",lit(silver_run_id))
                    
    )
    upsert_to_silver(products_cleaned,"novacart.silver.products_cleaned","product_id")
    products_validated = (
    products_cleaned
    .withColumn(
        "to_be_verified_by_products_team",
        when(
            col("product_name").isNull(),
            "verify_product_name"
        )
        .when(
            col("category").isNull(),
            "verify_category"
        )
        .when(
            (col("price").isNull()) | (col("price") <= 0),
            "verify_price"
        )
        .otherwise("No Issues")
    )
    .withColumn(
        "check_product_price",
        when(
            (col("price").isNull()) | (col("price") <= 0),
            "verify_price"
        )
        .otherwise("No Issues")
    )
)
    products_good=products_validated.filter(

        (col("to_be_verified_by_products_team")=="No Issues") &
        (col("check_product_price")=="No Issues")
    )
    if "price_raw" in products_good.columns:
        products_good=products_good.drop("price_raw")

    products_bad=products_validated.filter(

        (col("to_be_verified_by_products_team")!="No Issues") |
        (col("check_product_price")=="verify_price")
        ).withColumn("quarantine_ts",current_timestamp())
    upsert_to_silver(products_good,"novacart.silver.products_transformed","product_id")
    products_bad.write.format("delta").mode("append").saveAsTable("novacart.silver.products_quarantine")
    mx_ingested=products_inc.agg(max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run=products_inc.filter(col("bronze_ingested_at")==lit(mx_ingested)).agg(max("bronze_run_id").alias("mx")).collect()[0]["mx"]
    upsert_silver_control("products",mx_run,mx_ingested,products_good.count())
else:
    print("no new products bronze rows for silver")
    upsert_silver_control(
        "products",None,last_products_ingested_at,products_inc_count

    )

     

    

    

In [0]:
payments_inc,last_payments_ingested_at=get_incremental_bronze("novacart.bronze.payments_raw","payments")
print("payments last processed bronze ingested at=",last_payments_ingested_at)

payments_inc_count=payments_inc.count()
print(f"payments rows_to_process_in_silver={payments_inc_count}")

if payments_inc.count()>0:
    payments_window=Window.partitionBy("payment_id").orderBy(

        col("processed_at").cast("timestamp").desc(),col("bronze_ingested_at").desc()
    )

    payments_cleaned=(
        payments_inc
        .withColumn("payment_status",upper(trim(col("payment_status"))))
        .withColumn("payment_status",when(col("payment_status")=="",lit(None)).otherwise(col("payment_status")))
        .withColumn("paid_amount",trim(col("paid_amount")))
        .withColumn("paid_amount",regexp_replace(col("paid_amount"),r"\$",""))
        .withColumn("paid_amount",regexp_replace(col("paid_amount"),r",","."))
        .withColumn("paid_amount",regexp_replace(col("paid_amount"),r"\s+",""))
        .withColumn("paid_amount",expr("try_cast(paid_amount as double)"))
        .withColumn("processed_at",to_timestamp("processed_at"))
        .withColumn("row_rank",row_number().over(payments_window))
        .filter(col("row_rank")==1)
        .drop("row_rank")
        .withColumn("silver_run_id",lit(silver_run_id))
    )
    upsert_to_silver(payments_cleaned,"novacart.silver.payments_cleaned","payment_id")
    payments_validated=(
        payments_cleaned
        .withColumn(
            "to_be_verified_by_payments_team",
             when(col("order_id").isNull(),"verify_order_id")
            .when(col("payment_status").isNull(),"verify_payment_status")
            .when(col("paid_amount").isNull() | (col("paid_amount")<=0),"verify_paid_amount")
            .otherwise("No Issues")
            
        )
        .withColumn(
            "check_paid_amount",
            when(col("paid_amount").isNull()|(col("paid_amount")<=0),lit(True)).otherwise(lit(False))   
    )
    )
    payments_good=payments_validated.filter(col("to_be_verified_by_payments_team")=="No Issues")
    payments_bad=payments_validated.filter(col("to_be_verified_by_payments_team")!="No Issues").withColumn("guarantine_ts",current_timestamp())
    upsert_to_silver(payments_good,"novacart.silver.payments_transformed","payment_id")
    payments_bad.write.format("delta").mode("append").saveAsTable("novacart.silver.payments_quarantine")
    mx_ingested=payments_inc.agg(max("bronze_ingested_at").alias("mx")).collect()[0]["mx"]
    mx_run=payments_inc.filter(col("bronze_ingested_at")==lit(mx_ingested)).agg(max("bronze_run_id").alias("mx")).collect()[0]["mx"]
    upsert_silver_control("payments",mx_run,mx_ingested,payments_good.count())
else:
    print('no new payments bronze rows for silver.')
    upsert_silver_control("payments",None,last_payments_ingested_at,payments_inc_count)


In [0]:
#spark.sql("truncate table novacart.silver.processing_control")

In [0]:
%sql
-- select count(*) from novacart.silver.payments_cleaned;
-- select count(*) from novacart.silver.payments_quarantine;
-- select count(*) from novacart.silver.payments_transformed;
select count(*) from novacart.bronze.payments_raw


In [0]:
print("Products transformed count :", spark.sql("SELECT COUNT(*) FROM novacart.silver.products_transformed").collect()[0][0])

print("Orders transformed count   :", spark.sql("SELECT COUNT(*) FROM novacart.silver.orders_transformed").collect()[0][0])

print("Payments transformed count :", spark.sql("SELECT COUNT(*) FROM novacart.silver.payments_transformed").collect()[0][0])

display(
    spark.table("novacart.silver.processing_control")
         .orderBy("entity_name")
)

In [0]:
%sql
select * from novacart.silver.processing_control